In [1]:
%load_ext autoreload
%autoreload 2

import torch
import pandas as pd
import sys, os
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import copy
import warnings
import rpy2
from utils import *
from frengression import *
device = torch.device('cpu')


In [2]:
df = pd.read_csv('fakedataset_simcausal_complex.csv')

print(df.head())
df.info()

# Dummy data
s = df[['L1_0']].to_numpy(dtype=float)
x = df[['A1_0','A1_1','A1_2']].to_numpy(dtype=float)
y = df[['Y_1','Y_2','Y_3']].to_numpy(dtype=float)
z = df[['L2_0','L2_1','L2_2']].to_numpy(dtype=float)

s_tr = torch.tensor(s, dtype=torch.float32)
x_tr = torch.tensor(x, dtype=torch.int32)
y_tr = torch.tensor(y, dtype=torch.float32)
z_tr = torch.tensor(z, dtype=torch.float32)

   Unnamed: 0  ID  L2_0  L1_0  A1_0  Y_1  L2_1  A1_1  Y_2  L2_2  A1_2  Y_3  \
0           1   1     0     0     0    0     0     0    0     0     0    0   
1           2   2     0     0     0    0     0     0    0     0     0    0   
2           3   3     0     0     0    0     0     0    0     1     0    0   
3           4   4     0     0     0    0     0     0    0     0     0    0   
4           5   5     0     0     0    0     0     0    0     0     0    0   

   L2_3  A1_3  
0     0     0  
1     0     0  
2     1     0  
3     0     0  
4     0     0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  10000 non-null  int64
 1   ID          10000 non-null  int64
 2   L2_0        10000 non-null  int64
 3   L1_0        10000 non-null  int64
 4   A1_0        10000 non-null  int64
 5   Y_1         10000 non-null  int64
 6   L2_1      

In [3]:
model = FrengressionSeq(x_dim=1, y_dim=1, z_dim=1, T=3, s_dim = 1, noise_dim=1, 
                        num_layer=3, hidden_dim=100,#100 
                        device=device, x_binary = True, s_in_predict=True, y_binary=True)
model.train_xz(s=s_tr, x=x_tr,z=z_tr,num_iters=100, lr=1e-4, print_every_iter=1000)
model.train_e(s=s_tr, x=x_tr,z=z_tr,num_iters=100, lr=1e-4, print_every_iter=1000)
model.train_y(s=s_tr, x=x_tr,z=z_tr, y=y_tr, num_iters=100, lr=1e-4, print_every_iter=1000)

Epoch 1: loss 0.7204, loss1 0.9332, loss2 0.4256
Epoch 1: loss 0.4630, loss1 0.4966, loss2 0.0671
Epoch 1: loss 2.3991,	loss_y 0.8604, 0.8687, 0.0166,	loss_eta 1.5387, 1.6090, 0.1407


In [4]:
y_margin_sample=model.sample_causal_margin(s=torch.tensor([[0]],dtype=torch.float32),
                                            x = torch.tensor([[1]*5],dtype=torch.int32),
                                            sample_size=1000)

In [5]:
def five_point_summary(arr):
  arr = np.asarray(arr)
  return {
  "min": np.nanmin(arr),
  "q1": np.nanpercentile(arr, 25),
  "median": np.nanpercentile(arr, 50),
  "q3": np.nanpercentile(arr, 75),
  "max": np.nanmax(arr)
  }
print(five_point_summary(y_margin_sample[0]))
print(y_tr[0])
print(five_point_summary(y_margin_sample[1]))
print(five_point_summary(y_margin_sample[2]))
print(len(y_margin_sample[2]))

{'min': 0, 'q1': 0.0, 'median': 0.0, 'q3': 0.0, 'max': 0}
tensor([0., 0., 0.])
{'min': 0, 'q1': 0.0, 'median': 0.0, 'q3': 0.0, 'max': 0}
{'min': 0, 'q1': 0.0, 'median': 0.0, 'q3': 0.0, 'max': 0}
1


In [7]:
model.sample_joint(s=s_tr,sample_size=100)

(tensor([[0., 1., 0.],
         [1., 1., 0.],
         [0., 0., 0.],
         ...,
         [0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]]),
 tensor([[ 0.0699, -0.2229,  0.8774],
         [-0.4444,  0.0026,  0.3683],
         [ 0.0030,  0.3110,  0.2287],
         ...,
         [-0.2009, -0.1690,  0.2192],
         [ 0.5107, -0.2586,  0.0685],
         [-0.0894,  0.4703,  0.1046]]),
 tensor([[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]], dtype=torch.int32))